# 4.2 Importance & Missing Values

## Table of Contents
- [4.2.1 Why Data Prep is Crucial](#421-why-data-prep-is-crucial)
- [4.2.2 Handling Missing Values](#422-handling-missing-values)
- [4.2.3 Detecting Missing Values](#423-detecting-missing-values)
- [Knowledge Check](#knowledge-check)
- [Mini-Challenges](#mini-challenges)
- [Practical Connections](#practical-connections)

## Introduction

Now that we've learned how to evaluate models, let's turn our attention to the crucial step of preparing data *before* modeling. Data preparation is one of the most important (and often time-consuming) steps in the machine learning workflow. In this notebook, we'll focus on understanding the importance of data preparation and techniques for handling missing values.

First, let's import the necessary libraries:

In [ ]:
python
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 4.2.1 Why Data Prep is Crucial

### Garbage In, Garbage Out
Machine learning models are only as good as the data they are trained on. Poor quality data leads to poor models, regardless of the algorithm's sophistication.

### Algorithm Requirements
Many algorithms have specific requirements regarding data format (e.g., numerical inputs only) or scale (e.g., distance-based algorithms like KNN, or algorithms using gradient descent like Logistic Regression/Neural Networks).

### Improving Performance
Proper preprocessing can significantly improve model accuracy, robustness, and training speed.

### Key Tasks in Data Preparation
1. **Handling missing values**: Imputing or removing missing data
2. **Scaling numerical features**: Bringing features to similar scales
3. **Encoding categorical features**: Converting text categories to numbers
4. **Feature engineering**: Creating new informative features
5. **Handling outliers**: Detecting and dealing with extreme values
6. **Addressing data imbalance**: When classes have significantly different frequencies

### Impact of Data Preparation
Let's visualize the potential impact of proper data preparation:

In [ ]:
# Create a synthetic example to demonstrate the impact of data preparation
np.random.seed(42)

# Generate unscaled data with different ranges
feature1 = np.random.normal(0, 1, 100)  # Mean 0, std 1
feature2 = np.random.normal(0, 10, 100)  # Mean 0, std 10
feature3 = np.random.normal(1000, 500, 100)  # Mean 1000, std 500

# Create a DataFrame
data = pd.DataFrame({
    'Feature1': feature1,
    'Feature2': feature2,
    'Feature3': feature3
})

# Visualize the unscaled data
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.boxplot(data=data)
plt.title('Unscaled Features (Original Data)')
plt.ylabel('Value')

# Scale the data
scaler = StandardScaler()
data_scaled = pd.DataFrame(
    scaler.fit_transform(data),
    columns=data.columns
)

# Visualize the scaled data
plt.subplot(1, 2, 2)
sns.boxplot(data=data_scaled)
plt.title('Scaled Features (After Preprocessing)')
plt.ylabel('Value')

plt.tight_layout()
plt.show()

# Print statistics before and after scaling
print("Statistics before scaling:")
print(data.describe().round(2))
print("\nStatistics after scaling:")
print(data_scaled.describe().round(2))

As we can see from the visualization, scaling brings all features to a similar range, which can significantly improve the performance of many machine learning algorithms.

## 4.2.2 Handling Missing Values

### What are Missing Values?
Missing values are data points that are not available for a particular feature in one or more observations. They are often represented as:
- `NaN` (Not a Number) in pandas/numpy
- `None` in Python
- Sometimes specific placeholders (e.g., 999, -999, '?', 'Unknown', blank spaces)

### Why are Missing Values a Problem?
1. Most ML algorithms cannot handle missing values directly and will raise errors
2. They represent lost information that could be valuable for prediction
3. They can introduce bias if not handled properly
4. They can lead to incorrect analysis and conclusions

### Common Strategies for Handling Missing Values

#### 1. Deletion Methods
* **Listwise Deletion (Row Deletion):** 
  - Remove entire rows containing any missing values
  - Simple to implement
  - Can lead to significant data loss if many rows have missing values
  - Generally okay if only a tiny fraction of rows are affected (< 5%)

* **Column Deletion:** 
  - Remove entire columns (features) if they have a very high percentage of missing values (e.g., >50-70%) 
  - Useful when a feature is deemed not critical or too sparse to be informative
  - Use with caution, as you lose potential information

#### 2. Imputation Methods (Filling Values)
Replace missing values with estimated or calculated values. More common than deletion as it preserves data.

##### For Numerical Features:
* **Mean Imputation:** 
  - Replace missing values with the mean of the column
  - Simple to implement
  - Sensitive to outliers

* **Median Imputation:** 
  - Replace missing values with the median of the column
  - More robust to outliers than the mean
  - Often a good default for skewed data

* **Mode Imputation:** 
  - Replace missing values with the mode (most frequent value)
  - Can also be used for numerical data, though less common than mean/median

* **Advanced methods:**
  - **K-Nearest Neighbors (KNN) Imputation:** Impute values based on similar observations
  - **Regression Imputation:** Predict missing values using other features
  - **Multiple Imputation:** Generate multiple complete datasets and combine results

##### For Categorical Features:
* **Mode Imputation:** 
  - Replace missing values with the mode (most frequent category)
  - Simple and common

* **Create a "Missing" Category:** 
  - Treat missingness as its own category (e.g., fill `NaN` with "Missing" or "Unknown")
  - This can sometimes capture meaningful information if the reason for missingness is informative

> **Important Note:** Imputation should generally be done *after* splitting data into training and test sets. You should calculate the value to impute (e.g., mean, median, mode) *only* from the **training set** and then use that same value to fill missing values in *both* the training and test sets. This prevents data leakage from the test set into the training process.

Let's implement and compare some of these methods:

In [ ]:
# Create a dataset with missing values
np.random.seed(42)
n_samples = 1000
n_features = 5

# Create a DataFrame with random values
data = pd.DataFrame(
    np.random.randn(n_samples, n_features),
    columns=[f'feature_{i+1}' for i in range(n_features)]
)

# Introduce missing values (about 10% of the data)
for col in data.columns:
    mask = np.random.random(n_samples) < 0.1
    data.loc[mask, col] = np.nan

# Print the first 5 rows
print("Dataset with missing values:")
print(data.head())

# Split the data into training and test sets
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Count missing values in each dataset
train_missing = train_data.isnull().sum()
test_missing = test_data.isnull().sum()

print("\nMissing values in training set:")
print(train_missing)
print("\nMissing values in test set:")
print(test_missing)

# 1. Mean Imputation (done correctly with fit on train, transform on both)
mean_imputer = SimpleImputer(strategy='mean')
mean_imputer.fit(train_data)  # Fit on training data only

# Transform both datasets
train_imputed_mean = pd.DataFrame(
    mean_imputer.transform(train_data),
    columns=train_data.columns,
    index=train_data.index
)

test_imputed_mean = pd.DataFrame(
    mean_imputer.transform(test_data),
    columns=test_data.columns,
    index=test_data.index
)

# 2. Median Imputation
median_imputer = SimpleImputer(strategy='median')
median_imputer.fit(train_data)  # Fit on training data only

# Transform both datasets
train_imputed_median = pd.DataFrame(
    median_imputer.transform(train_data),
    columns=train_data.columns,
    index=train_data.index
)

test_imputed_median = pd.DataFrame(
    median_imputer.transform(test_data),
    columns=test_data.columns,
    index=test_data.index
)

# Compare original vs imputed values for feature_1
plt.figure(figsize=(12, 6))

# Original distribution (non-missing values only)
plt.subplot(1, 3, 1)
sns.histplot(train_data['feature_1'].dropna(), kde=True)
plt.title('Original (Non-Missing Values)')
plt.xlabel('Value')

# Distribution after mean imputation
plt.subplot(1, 3, 2)
sns.histplot(train_imputed_mean['feature_1'], kde=True)
plt.title('After Mean Imputation')
plt.xlabel('Value')

# Distribution after median imputation
plt.subplot(1, 3, 3)
sns.histplot(train_imputed_median['feature_1'], kde=True)
plt.title('After Median Imputation')
plt.xlabel('Value')

plt.tight_layout()
plt.show()

# Calculate and print imputation values
mean_values = mean_imputer.statistics_
median_values = median_imputer.statistics_

# Create a comparison DataFrame
imputation_comparison = pd.DataFrame({
    'Mean Imputation': mean_values,
    'Median Imputation': median_values
}, index=[f'feature_{i+1}' for i in range(n_features)])

print("\nImputation values used:")
print(imputation_comparison)

# Show the effect of data leakage (incorrect approach - for demonstration only)
print("\nDemonstrating Data Leakage Problem:")
# Incorrectly imputing by fitting on entire dataset
incorrect_imputer = SimpleImputer(strategy='mean')
incorrect_imputer.fit(pd.concat([train_data, test_data]))  # Incorrectly fitting on all data

# Values calculated from all data
incorrect_values = incorrect_imputer.statistics_

# Compare correct vs incorrect approach
comparison = pd.DataFrame({
    'Training Mean (Correct)': mean_values,
    'Full Dataset Mean (Incorrect)': incorrect_values,
    'Difference': mean_values - incorrect_values
}, index=[f'feature_{i+1}' for i in range(n_features)])

print(comparison.round(6))
print("\nThis small difference might seem insignificant, but it represents data leakage")
print("where information from the test set influences the training process.")

## 4.2.3 Detecting Missing Values

Before we can handle missing values, we need to detect them. Let's explore some techniques for identifying missing values in a dataset:

In [ ]:
python
# Create a dataset with various types of missing values
data_missing = pd.DataFrame({
    'numeric_nan': [1.0, np.nan, 3.0, np.nan, 5.0],
    'numeric_none': [1.0, None, 3.0, None, 5.0],
    'numeric_missing_code': [1.0, -999.0, 3.0, -999.0, 5.0],
    'string_nan': ['a', np.nan, 'c', np.nan, 'e'],
    'string_none': ['a', None, 'c', None, 'e'],
    'string_missing_code': ['a', '?', 'c', 'Unknown', 'e']
})

print("Dataset with various missing value representations:")
print(data_missing)

# 1. Standard detection of NaN and None
missing_standard = data_missing.isnull().sum()
print("\nDetected missing values (standard method):")
print(missing_standard)

# 2. Visualizing missing values
plt.figure(figsize=(10, 6))
sns.heatmap(data_missing.isnull(), cmap='viridis', cbar=False, yticklabels=False)
plt.title('Missing Value Patterns')
plt.tight_layout()
plt.show()

# 3. Handling non-standard missing values (like -999, '?', 'Unknown')
# Replace specific values with NaN
data_cleaned = data_missing.copy()
data_cleaned['numeric_missing_code'].replace(-999.0, np.nan, inplace=True)
data_cleaned['string_missing_code'].replace(['?', 'Unknown'], np.nan, inplace=True)

# Check again
missing_after_cleaning = data_cleaned.isnull().sum()
print("\nDetected missing values after cleaning:")
print(missing_after_cleaning)

# Visualize again
plt.figure(figsize=(10, 6))
sns.heatmap(data_cleaned.isnull(), cmap='viridis', cbar=False, yticklabels=False)
plt.title('Missing Value Patterns (After Cleaning)')
plt.tight_layout()
plt.show()

# 4. Summarizing missing values
def missing_values_summary(df):
    # Count missing values
    missing = df.isnull().sum()
    
    # Calculate percentage
    missing_percent = 100 * missing / len(df)
    
    # Create summary DataFrame
    missing_summary = pd.DataFrame({
        'Missing Values': missing,
        'Percentage': missing_percent.round(2)
    })
    
    # Sort by percentage
    missing_summary = missing_summary.sort_values('Percentage', ascending=False)
    
    return missing_summary

print("\nMissing values summary:")
print(missing_values_summary(data_cleaned))

### Best Practices for Handling Missing Values

1. **Understand Why Data is Missing**
   - **Missing Completely at Random (MCAR)**: Missing values occur entirely at random with no pattern
   - **Missing at Random (MAR)**: Missing values have patterns related to observed data
   - **Missing Not at Random (MNAR)**: Missing values have patterns related to unobserved data
   - The reason for missingness can inform your imputation strategy

2. **Document Your Missing Value Strategy**
   - Record how many values were missing and how they were handled
   - This is essential for reproducibility and understanding model limitations

3. **Consider Domain Knowledge**
   - In some fields, missing values might have specific meanings
   - For example, in medical data, missing lab tests may indicate the test wasn't needed

4. **Use a Data Leakage Prevention Workflow**
   1. Split your data into training and test sets first
   2. Fit imputers only on the training data
   3. Apply the same imputation to both training and test data
   4. Validate that your missing value strategy doesn't introduce bias

5. **Try Multiple Approaches**
   - Different imputation strategies might work better for different datasets
   - Consider evaluating model performance with various imputation methods

## Knowledge Check

1. What is data leakage in the context of imputing missing values?
2. Why might median imputation be preferred over mean imputation in some cases?
3. In what situations would creating a "Missing" category be better than imputing values?
4. Why is it important to detect and handle missing values before modeling?
5. If you have a feature with 75% missing values, what would be a reasonable approach to handle it?

## Mini-Challenges

### Challenge 1: Missing Value Detection
A dataset has the following "missing value" indicators: -999, "N/A", "unknown", and blank cells. Write code to identify and replace all these with NaN.

In [ ]:
python
# Starter code
import pandas as pd
import numpy as np

# Create sample data
data = pd.DataFrame({
    'A': [1, -999, 3, 4, 5],
    'B': ['apple', 'N/A', 'orange', 'banana', 'unknown'],
    'C': ['red', 'blue', '', 'green', 'yellow']
})

# Your code here to replace all missing value indicators with NaN
# Hint: Use .replace() for specific values

### Challenge 2: Imputation Strategies
For the dataset below, implement and compare three different imputation strategies:
1. Mean imputation
2. Median imputation
3. Most frequent value imputation

In [ ]:
python
# Starter code
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# Sample dataset
np.random.seed(42)
data = pd.DataFrame({
    'Age': [25, 30, np.nan, 40, np.nan, 35, 42, np.nan],
    'Income': [50000, np.nan, 60000, np.nan, 75000, 65000, np.nan, 80000],
    'Experience': [2, 5, np.nan, 10, 15, np.nan, 12, 20]
})

# Your code here to implement and compare imputation strategies

### Challenge 3: Advanced - Build a Pipeline
Create a sklearn Pipeline that:
1. Splits the data into train and test sets
2. Handles missing values using median imputation
3. Implements a simple model with the processed data

## Practical Connections

- **Healthcare**: Missing values in patient medical records might indicate tests that weren't performed because the doctor deemed them unnecessary, which itself can be informative.

- **Customer Data**: In customer databases, missing demographic information might be correlated with certain customer behaviors or segments, making the pattern of missingness potentially valuable information.

- **Sensor Networks**: In IoT sensor networks, missing readings could indicate sensor malfunctions, environmental conditions, or power outages that might be relevant to the analysis.

- **Financial Data**: Missing financial data points might indicate regulatory issues, financial distress, or deliberate non-disclosure, all of which could be valuable signals.

- **Survey Data**: Survey responses often have missing values when participants skip questions, which might reveal sensitive questions or respondent fatigue patterns.

- **Time Series**: In time series data, missing values can represent service outages, holidays, or other temporal patterns that should be preserved rather than imputed away.